# Sentinel-2 NDCI Analysis and Improved Calibration

This notebook analyzes the Sentinel-2 NDCI data distribution and develops better calibration approaches.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## Load and Analyze NDCI Data Distribution

In [ ]:
# Load Sentinel-2 NDCI data
ukl_sentinel = pd.read_csv('Klamath_S2_NDCI_500m.csv')
detroit_sentinel = pd.read_csv('Detroit_S2_NDCI_500m.csv')

# Convert dates
ukl_sentinel['date'] = pd.to_datetime(ukl_sentinel['date'])
detroit_sentinel['date'] = pd.to_datetime(detroit_sentinel['date'])

print("Data loaded successfully!")
print(f"\nUpper Klamath Lake NDCI statistics:")
print(ukl_sentinel['ndci'].describe())
print(f"\nDetroit Lake NDCI statistics:")
print(detroit_sentinel['ndci'].describe())

In [ ]:
# Visualize NDCI distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# UKL histogram
axes[0, 0].hist(ukl_sentinel['ndci'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Upper Klamath Lake NDCI Distribution')
axes[0, 0].set_xlabel('NDCI')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(ukl_sentinel['ndci'].median(), color='red', linestyle='--', label=f'Median: {ukl_sentinel["ndci"].median():.3f}')
axes[0, 0].legend()

# Detroit histogram
axes[0, 1].hist(detroit_sentinel['ndci'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('Detroit Lake NDCI Distribution')
axes[0, 1].set_xlabel('NDCI')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(detroit_sentinel['ndci'].median(), color='red', linestyle='--', label=f'Median: {detroit_sentinel["ndci"].median():.3f}')
axes[0, 1].legend()

# Time series comparison
axes[1, 0].scatter(ukl_sentinel['date'], ukl_sentinel['ndci'], alpha=0.5, s=10, label='UKL')
axes[1, 0].set_title('Upper Klamath Lake NDCI Time Series')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('NDCI')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].scatter(detroit_sentinel['date'], detroit_sentinel['ndci'], alpha=0.5, s=10, color='green', label='Detroit')
axes[1, 1].set_title('Detroit Lake NDCI Time Series')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('NDCI')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Compare distributions
print("\nDistribution Comparison:")
print(f"UKL NDCI range: {ukl_sentinel['ndci'].min():.3f} to {ukl_sentinel['ndci'].max():.3f}")
print(f"Detroit NDCI range: {detroit_sentinel['ndci'].min():.3f} to {detroit_sentinel['ndci'].max():.3f}")
print(f"UKL NDCI std dev: {ukl_sentinel['ndci'].std():.3f}")
print(f"Detroit NDCI std dev: {detroit_sentinel['ndci'].std():.3f}")

## Compare Different NDCI to Chlorophyll Transformations

In [ ]:
def ndci_to_chl_quadratic(ndci):
    """Original quadratic transformation"""
    return 14.039 + 86.115 * ndci + 194.325 * ndci**2

def ndci_to_chl_exponential(ndci):
    """Exponential transformation for better dynamic range"""
    # Adjusted for typical lake NDCI values
    return 10 * np.exp(3 * (ndci + 0.2))  # Shift to make negative NDCI values work

def ndci_to_chl_linear(ndci, slope=100, intercept=20):
    """Simple linear transformation"""
    return slope * ndci + intercept

def ndci_to_chl_power(ndci):
    """Power transformation for positive NDCI"""
    # Shift NDCI to ensure positive values
    ndci_shifted = ndci + 0.5  # Shift to make all values positive
    return 30 * (ndci_shifted ** 1.5)

# Test transformations on NDCI range
ndci_test = np.linspace(-0.3, 0.6, 100)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Quadratic (original)
chl_quad = ndci_to_chl_quadratic(ndci_test)
axes[0, 0].plot(ndci_test, chl_quad, 'b-', linewidth=2)
axes[0, 0].set_title('Quadratic Transformation (Original)')
axes[0, 0].set_xlabel('NDCI')
axes[0, 0].set_ylabel('Chlorophyll (µg/L)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim(0, 200)

# Exponential
chl_exp = ndci_to_chl_exponential(ndci_test)
axes[0, 1].plot(ndci_test, chl_exp, 'g-', linewidth=2)
axes[0, 1].set_title('Exponential Transformation')
axes[0, 1].set_xlabel('NDCI')
axes[0, 1].set_ylabel('Chlorophyll (µg/L)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 200)

# Linear
chl_linear = ndci_to_chl_linear(ndci_test)
axes[1, 0].plot(ndci_test, chl_linear, 'r-', linewidth=2)
axes[1, 0].set_title('Linear Transformation')
axes[1, 0].set_xlabel('NDCI')
axes[1, 0].set_ylabel('Chlorophyll (µg/L)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim(-10, 100)

# Power
chl_power = ndci_to_chl_power(ndci_test)
axes[1, 1].plot(ndci_test, chl_power, 'm-', linewidth=2)
axes[1, 1].set_title('Power Transformation')
axes[1, 1].set_xlabel('NDCI')
axes[1, 1].set_ylabel('Chlorophyll (µg/L)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0, 100)

plt.suptitle('Comparison of NDCI to Chlorophyll Transformations', fontsize=14)
plt.tight_layout()
plt.show()

## Apply Transformations to Real Data

In [ ]:
# Apply different transformations to Detroit Lake data
detroit_sentinel_clean = detroit_sentinel[detroit_sentinel['ndci'].notna()].copy()

detroit_sentinel_clean['chl_quadratic'] = ndci_to_chl_quadratic(detroit_sentinel_clean['ndci'])
detroit_sentinel_clean['chl_exponential'] = ndci_to_chl_exponential(detroit_sentinel_clean['ndci'])
detroit_sentinel_clean['chl_linear'] = ndci_to_chl_linear(detroit_sentinel_clean['ndci'])
detroit_sentinel_clean['chl_power'] = ndci_to_chl_power(detroit_sentinel_clean['ndci'])

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Quadratic
axes[0, 0].scatter(detroit_sentinel_clean['date'], detroit_sentinel_clean['chl_quadratic'], 
                   alpha=0.5, s=10, color='blue')
axes[0, 0].set_title(f'Quadratic: Range {detroit_sentinel_clean["chl_quadratic"].min():.1f} - {detroit_sentinel_clean["chl_quadratic"].max():.1f} µg/L')
axes[0, 0].set_ylabel('Chlorophyll (µg/L)')
axes[0, 0].grid(True, alpha=0.3)

# Exponential
axes[0, 1].scatter(detroit_sentinel_clean['date'], detroit_sentinel_clean['chl_exponential'], 
                   alpha=0.5, s=10, color='green')
axes[0, 1].set_title(f'Exponential: Range {detroit_sentinel_clean["chl_exponential"].min():.1f} - {detroit_sentinel_clean["chl_exponential"].max():.1f} µg/L')
axes[0, 1].set_ylabel('Chlorophyll (µg/L)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 100)

# Linear
axes[1, 0].scatter(detroit_sentinel_clean['date'], detroit_sentinel_clean['chl_linear'], 
                   alpha=0.5, s=10, color='red')
axes[1, 0].set_title(f'Linear: Range {detroit_sentinel_clean["chl_linear"].min():.1f} - {detroit_sentinel_clean["chl_linear"].max():.1f} µg/L')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Chlorophyll (µg/L)')
axes[1, 0].grid(True, alpha=0.3)

# Power
axes[1, 1].scatter(detroit_sentinel_clean['date'], detroit_sentinel_clean['chl_power'], 
                   alpha=0.5, s=10, color='magenta')
axes[1, 1].set_title(f'Power: Range {detroit_sentinel_clean["chl_power"].min():.1f} - {detroit_sentinel_clean["chl_power"].max():.1f} µg/L')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Chlorophyll (µg/L)')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Detroit Lake: Different NDCI→Chlorophyll Transformations', fontsize=14)
plt.tight_layout()
plt.show()

# Statistics
print("\nTransformation Statistics for Detroit Lake:")
print(f"Quadratic:   Mean={detroit_sentinel_clean['chl_quadratic'].mean():.1f}, Std={detroit_sentinel_clean['chl_quadratic'].std():.1f}")
print(f"Exponential: Mean={detroit_sentinel_clean['chl_exponential'].mean():.1f}, Std={detroit_sentinel_clean['chl_exponential'].std():.1f}")
print(f"Linear:      Mean={detroit_sentinel_clean['chl_linear'].mean():.1f}, Std={detroit_sentinel_clean['chl_linear'].std():.1f}")
print(f"Power:       Mean={detroit_sentinel_clean['chl_power'].mean():.1f}, Std={detroit_sentinel_clean['chl_power'].std():.1f}")

## Direct NDCI Calibration (Skip Initial Transformation)

In [ ]:
# Load synthetic in situ data for demonstration
# This would be replaced with real in situ data
np.random.seed(42)
dates = pd.date_range('2018-01-01', '2024-12-31', freq='7D')
day_of_year = dates.dayofyear
seasonal_factor = 1 + 0.8 * np.sin((day_of_year - 80) * 2 * np.pi / 365)
base_chl = 30 * seasonal_factor
noise = np.random.lognormal(mean=0, sigma=0.5, size=len(dates))
chl_values = np.clip(base_chl * noise, 5, 200)

insitu_df = pd.DataFrame({
    'date': dates,
    'chlorophyll_ugL': chl_values
})

print("Created synthetic in situ data for demonstration")
print(f"In situ range: {insitu_df['chlorophyll_ugL'].min():.1f} - {insitu_df['chlorophyll_ugL'].max():.1f} µg/L")

In [ ]:
# Match Sentinel NDCI directly with in situ chlorophyll
# This demonstrates direct calibration without initial transformation

def match_data(satellite_df, insitu_df, tolerance_days=5):
    """Match satellite and in situ observations"""
    matches = []
    
    for _, sat_row in satellite_df.iterrows():
        sat_date = sat_row['date']
        time_diff = np.abs((insitu_df['date'] - sat_date).dt.days)
        within_tolerance = time_diff <= tolerance_days
        
        if within_tolerance.any():
            closest_idx = time_diff[within_tolerance].idxmin()
            insitu_row = insitu_df.loc[closest_idx]
            
            matches.append({
                'ndci': sat_row['ndci'],
                'insitu_chl': insitu_row['chlorophyll_ugL'],
                'days_diff': time_diff[closest_idx]
            })
    
    return pd.DataFrame(matches)

# Match UKL data
ukl_matches = match_data(ukl_sentinel[ukl_sentinel['ndci'].notna()], insitu_df, 5)
print(f"Matched {len(ukl_matches)} observations")

if len(ukl_matches) > 0:
    # Fit different models directly from NDCI to chlorophyll
    X = ukl_matches['ndci'].values.reshape(-1, 1)
    y = ukl_matches['insitu_chl'].values
    
    # Linear model
    linear_model = LinearRegression()
    linear_model.fit(X, y)
    linear_r2 = linear_model.score(X, y)
    
    # Polynomial model
    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=2)),
        ('linear', LinearRegression())
    ])
    poly_model.fit(X, y)
    poly_r2 = poly_model.score(X, y)
    
    # Robust linear model
    robust_model = HuberRegressor()
    robust_model.fit(X, y)
    robust_r2 = robust_model.score(X, y)
    
    print(f"\nDirect NDCI → Chlorophyll Calibration Results:")
    print(f"Linear R²: {linear_r2:.3f}")
    print(f"Polynomial R²: {poly_r2:.3f}")
    print(f"Robust Linear R²: {robust_r2:.3f}")
    
    # Plot calibrations
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    ndci_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
    
    # Linear
    axes[0].scatter(X, y, alpha=0.5)
    axes[0].plot(ndci_range, linear_model.predict(ndci_range), 'r-', linewidth=2)
    axes[0].set_xlabel('NDCI')
    axes[0].set_ylabel('Chlorophyll (µg/L)')
    axes[0].set_title(f'Linear (R²={linear_r2:.3f})')
    axes[0].grid(True, alpha=0.3)
    
    # Polynomial
    axes[1].scatter(X, y, alpha=0.5)
    axes[1].plot(ndci_range, poly_model.predict(ndci_range), 'g-', linewidth=2)
    axes[1].set_xlabel('NDCI')
    axes[1].set_ylabel('Chlorophyll (µg/L)')
    axes[1].set_title(f'Polynomial (R²={poly_r2:.3f})')
    axes[1].grid(True, alpha=0.3)
    
    # Robust
    axes[2].scatter(X, y, alpha=0.5)
    axes[2].plot(ndci_range, robust_model.predict(ndci_range), 'm-', linewidth=2)
    axes[2].set_xlabel('NDCI')
    axes[2].set_ylabel('Chlorophyll (µg/L)')
    axes[2].set_title(f'Robust Linear (R²={robust_r2:.3f})')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('Direct NDCI to Chlorophyll Calibration', fontsize=14)
    plt.tight_layout()
    plt.show()

## Recommendations

In [ ]:
print("="*60)
print("RECOMMENDATIONS FOR SENTINEL-2 CALIBRATION")
print("="*60)
print()
print("1. DIRECT CALIBRATION:")
print("   - Calibrate NDCI directly to in situ chlorophyll")
print("   - Skip the initial NDCI→Chl transformation step")
print("   - This preserves the natural variation in the data")
print()
print("2. USE LINEAR OR POLYNOMIAL MODEL:")
print("   - Linear: Chl = a * NDCI + b")
print("   - Polynomial: Chl = a * NDCI² + b * NDCI + c")
print("   - These preserve more variation than complex transformations")
print()
print("3. CONSIDER SEASONAL MODELS:")
print("   - Separate calibrations for different seasons")
print("   - NDCI response varies with algae species composition")
print()
print("4. OUTLIER HANDLING:")
print("   - Use robust regression (Huber or RANSAC)")
print("   - Filter extreme NDCI values (<-0.5 or >0.8)")
print()
print("5. VALIDATION:")
print("   - Compare Sentinel-2 and MODIS during overlapping periods")
print("   - Ensure similar seasonal patterns")
print("="*60)